# Re-upload Aug16_4_ref (got cut off last time)
Only processes the 3rd session — Aug16_1 and Aug16_3 are already on HuggingFace.

In [ ]:
!pip install -q huggingface_hub h5py

import os, re, shutil, requests, time
import numpy as np
import h5py
from huggingface_hub import HfApi, hf_hub_download, login

WILD_PASSWORD = "h!#ASe9018$%_rt#"
HF_DATASET_REPO = "JustAGeek/dloc-wild-fig10b"
HF_TOKEN = "YOUR_HF_WRITE_TOKEN_HERE"  # <-- paste your token

WORK_DIR = "/tmp/wild"
os.makedirs(WORK_DIR, exist_ok=True)
CHUNK = 300

login(token=HF_TOKEN)
api = HfApi()
print(f"Logged in as: {api.whoami()['name']}")
print(f"Disk free: {shutil.disk_usage('/tmp').free / 1e9:.1f} GB")

In [ ]:
def download_wild_file(sharepoint_url, password, dest_path):
    if os.path.exists(dest_path):
        print(f"  Already exists: {dest_path}")
        return True
    session = requests.Session()
    session.headers.update({'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'})
    print(f"  Authenticating...")
    r = session.get(sharepoint_url, allow_redirects=True)
    if r.status_code != 200:
        print(f"  ERROR: GET returned {r.status_code}")
        return False
    action = re.search(r'<form[^>]*action="([^"]+)"', r.text)
    if not action:
        print("  ERROR: No password form found")
        return False
    form_url = 'https://ucsdcloud-my.sharepoint.com' + action.group(1).replace('&amp;', '&')
    fields = {}
    for match in re.finditer(r'<input[^>]*name="([^"]+)"[^>]*value="([^"]*)"', r.text):
        fields[match.group(1)] = match.group(2)
    fields['txtPassword'] = password
    fields['btnSubmitPassword'] = 'Verify'
    print(f"  Submitting password...")
    r2 = session.post(form_url, data=fields, allow_redirects=False)
    if r2.status_code != 302 or 'Location' not in r2.headers:
        print(f"  ERROR: Auth failed (status {r2.status_code})")
        return False
    download_url = 'https://ucsdcloud-my.sharepoint.com' + r2.headers['Location']
    print(f"  Downloading...")
    r3 = session.get(download_url, stream=True, allow_redirects=True)
    if r3.status_code != 200:
        print(f"  ERROR: Download returned {r3.status_code}")
        return False
    total = int(r3.headers.get('Content-Length', 0))
    total_gb = total / 1e9
    print(f"  File size: {total_gb:.1f} GB")
    downloaded = 0
    t0 = time.time()
    with open(dest_path + '.partial', 'wb') as f:
        for chunk in r3.iter_content(chunk_size=64 * 1024 * 1024):
            f.write(chunk)
            downloaded += len(chunk)
            elapsed = time.time() - t0
            speed = downloaded / elapsed / 1e6
            pct = 100 * downloaded / total if total else 0
            print(f"  {downloaded/1e9:.1f}/{total_gb:.1f} GB ({pct:.0f}%) — {speed:.0f} MB/s", end='\r')
    print()
    os.rename(dest_path + '.partial', dest_path)
    print(f"  Saved: {dest_path} ({time.time()-t0:.0f}s)")
    return True

print("Functions defined.")

In [ ]:
# Download split index file
split_path = hf_hub_download(
    repo_id=HF_DATASET_REPO, repo_type="dataset",
    filename="data_split_idx/data_split_ids_jacobs_Aug16_4_ref.mat")
print(f"Split file: {split_path}")

# Download raw features
raw_path = os.path.join(WORK_DIR, "features_jacobs_Aug16_4_ref.mat")
out_path = os.path.join(WORK_DIR, "dataset_jacobs_Aug16_4_ref.mat")

print("\n[1/4] Downloading raw features...")
ok = download_wild_file(
    "https://ucsdcloud-my.sharepoint.com/:u:/g/personal/sayyalas_ucsd_edu/ETdyRRQe7UlJg5Aa5VsFf1ABsLj-aQWnRINB2VpHX_6XNw?download=1",
    WILD_PASSWORD, raw_path)
assert ok, "Download failed!"
print(f"  Disk free: {shutil.disk_usage('/tmp').free / 1e9:.1f} GB")

# Process
print("\n[2/4] Processing...")
with h5py.File(split_path, 'r') as sf:
    all_idx = []
    for key in ['fov_train_idx', 'non_fov_train_idx', 'fov_test_idx', 'non_fov_test_idx']:
        idx = np.array(sf[key]).flatten().astype(np.int64) - 1
        all_idx.append(idx)
        print(f"    {key}: {len(idx)} samples")
    all_idx = np.sort(np.concatenate(all_idx))
n = len(all_idx)
print(f"  Total: {n} samples")

with h5py.File(raw_path, 'r') as src:
    keys = list(src.keys())
    w_key = 'features_w_offset' if 'features_w_offset' in keys else 'features_with_offset'
    wo_key = 'features_wo_offset' if 'features_wo_offset' in keys else 'features_without_offset'
    feat_shape = src[w_key].shape
    lbl_shape = src['labels_gaussian_2d'].shape
    print(f"  Raw shapes: features={feat_shape}, labels={lbl_shape}")
    out_feat_shape = feat_shape[:-1] + (n,)
    out_lbl_shape = lbl_shape[:-1] + (n,)
    with h5py.File(out_path, 'w') as dst:
        ds_w = dst.create_dataset('features_w_offset', shape=out_feat_shape, dtype='float32', compression='gzip', chunks=True)
        ds_wo = dst.create_dataset('features_wo_offset', shape=out_feat_shape, dtype='float32', compression='gzip', chunks=True)
        ds_lbl = dst.create_dataset('labels_gaussian_2d', shape=out_lbl_shape, dtype='float32', compression='gzip', chunks=True)
        written = 0
        for start in range(0, n, CHUNK):
            end = min(start + CHUNK, n)
            chunk_idx = all_idx[start:end].tolist()
            bs = end - start
            ds_w[..., written:written+bs] = np.array(src[w_key][..., chunk_idx], dtype=np.float32)
            ds_wo[..., written:written+bs] = np.array(src[wo_key][..., chunk_idx], dtype=np.float32)
            ds_lbl[..., written:written+bs] = np.array(src['labels_gaussian_2d'][..., chunk_idx], dtype=np.float32)
            written += bs
            print(f"  Processed {written}/{n}", end='\r')
        print(f"  Processed {n}/{n}")

out_size = os.path.getsize(out_path) / 1e9
print(f"  Output: {out_size:.2f} GB")

# Delete raw
print("\n[3/4] Deleting raw file...")
os.remove(raw_path)
print(f"  Disk free: {shutil.disk_usage('/tmp').free / 1e9:.1f} GB")

# Upload
print(f"\n[4/4] Uploading ({out_size:.2f} GB)...")
api.upload_file(
    path_or_fileobj=out_path,
    path_in_repo="data/dataset_jacobs_Aug16_4_ref.mat",
    repo_id=HF_DATASET_REPO,
    repo_type="dataset")
print("DONE! dataset_jacobs_Aug16_4_ref.mat uploaded to HuggingFace.")